In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pprint import pprint

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

In [24]:
DATA_PATH = Path("/home/ubuntu/MLLM-interpretability/data/processed/training_data.csv")
TARGET = "correct_target"

In [25]:
MODEL1_FEATURES = ["source_lang", "target_lang"]

In [26]:
def split_features(df, feature_list):
    X = df[feature_list].copy()
    y = df[TARGET].copy()
    if y.dtype == "O":
        y = y.map({True: 1, False: 0, "true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    y = y.astype(float).astype(int)
    return X, y

def build_preprocess_pipe(X: pd.DataFrame) -> ColumnTransformer:
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    pre = ColumnTransformer([("num", num_pipe, numeric_cols), ("cat", cat_pipe, categorical_cols)], remainder="drop", verbose_feature_names_out=False)
    return pre

def train_and_evaluate(df, feature_list, model_name, random_state=42):
    X, y = split_features(df, feature_list)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=random_state)
    pre = build_preprocess_pipe(X_train)
    clf = LogisticRegression(penalty="l2", solver="saga", max_iter=5000, n_jobs=-1, class_weight="balanced", random_state=random_state)
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (proba >= 0.5).astype(int)
    roc_auc = roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
    pr_auc = average_precision_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
    precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)
    summary = {
        "model": model_name,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "positive_rate_train": float(np.mean(y_train)),
        "positive_rate_test": float(np.mean(y_test)),
        "ROC_AUC": float(roc_auc) if not np.isnan(roc_auc) else None,
        "PR_AUC": float(pr_auc) if not np.isnan(pr_auc) else None,
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
    }
    print(f"\n=== {model_name} ===")
    print("Summary:", summary)
    print("\nClassification Report:\n", report)
    return pipe, summary

In [27]:
df = pd.read_csv(DATA_PATH)
results = []

In [28]:
pipe1, s1 = train_and_evaluate(df, MODEL1_FEATURES, "Model 1: (source_lang, target_lang)")
results.append(s1)


=== Model 1: (source_lang, target_lang) ===
Summary: {'model': 'Model 1: (source_lang, target_lang)', 'n_train': 374, 'n_test': 94, 'positive_rate_train': 0.6016042780748663, 'positive_rate_test': 0.6063829787234043, 'ROC_AUC': 0.6014698909435752, 'PR_AUC': 0.6743797504246098, 'Precision': 0.71875, 'Recall': 0.40350877192982454, 'F1': 0.5168539325842697, 'TN': 28, 'FP': 9, 'FN': 34, 'TP': 23}

Classification Report:
               precision    recall  f1-score   support

           0       0.45      0.76      0.57        37
           1       0.72      0.40      0.52        57

    accuracy                           0.54        94
   macro avg       0.59      0.58      0.54        94
weighted avg       0.61      0.54      0.54        94



In [29]:
for r in results:
    print(r)

{'model': 'Model 1: (source_lang, target_lang)', 'n_train': 374, 'n_test': 94, 'positive_rate_train': 0.6016042780748663, 'positive_rate_test': 0.6063829787234043, 'ROC_AUC': 0.6014698909435752, 'PR_AUC': 0.6743797504246098, 'Precision': 0.71875, 'Recall': 0.40350877192982454, 'F1': 0.5168539325842697, 'TN': 28, 'FP': 9, 'FN': 34, 'TP': 23}


In [30]:
pipe1.named_steps["clf"].coef_

array([[ 2.35119459e-04, -2.10154859e-02,  8.84536783e-01,
        -2.30958003e-01,  3.65401322e-01, -2.70312317e-01,
        -4.95718949e-01,  1.86348351e-01, -2.60978923e-01,
        -1.10302084e-01, -2.61014896e-01,  2.34467536e-01,
        -2.02182143e-02]])

In [31]:
pipe1.named_steps["pre"].get_feature_names_out()

array(['source_lang_en', 'target_lang_de', 'target_lang_en',
       'target_lang_es', 'target_lang_fr', 'target_lang_he',
       'target_lang_hi', 'target_lang_id', 'target_lang_it',
       'target_lang_ja', 'target_lang_ko', 'target_lang_pt',
       'target_lang_zh'], dtype=object)

In [15]:
MODEL2_FEATURES = ['language_version_count', 'question_type', 'cooc_num_pairs', 'cooc_total_pairs', 'cooc_coverage_ratio', 'cooc_unseen_keywords_count', 'cooc_unseen_keywords_ratio', 'cooc_avg_pmi', 'cooc_max_pmi', 'cooc_min_pmi', 'cooc_std_pmi', 'target_avg_dep_depth', 'target_max_tree_depth', 'target_num_clauses', 'source_avg_dep_depth', 'source_max_tree_depth', 'source_num_clauses', 'source_lang_name', 'target_lang_name', 'source_family', 'source_genus', 'target_family', 'target_genus', 'source_script', 'source_syllables', 'source_wiki_size', 'target_script', 'target_syllables', 'target_wiki_size']

In [16]:
pipe2, s2 = train_and_evaluate(df, MODEL2_FEATURES, "Model 2: (rich feature set)")
results.append(s2)


=== Model 2: (rich feature set) ===
Summary: {'model': 'Model 2: (rich feature set)', 'n_train': 374, 'n_test': 94, 'positive_rate_train': 0.6016042780748663, 'positive_rate_test': 0.6063829787234043, 'ROC_AUC': 0.7477477477477478, 'PR_AUC': 0.827377860557543, 'Precision': 0.7413793103448276, 'Recall': 0.7543859649122807, 'F1': 0.7478260869565218, 'TN': 22, 'FP': 15, 'FN': 14, 'TP': 43}

Classification Report:
               precision    recall  f1-score   support

           0       0.61      0.59      0.60        37
           1       0.74      0.75      0.75        57

    accuracy                           0.69        94
   macro avg       0.68      0.67      0.68        94
weighted avg       0.69      0.69      0.69        94



In [18]:
results

[{'model': 'Model 1: (source_lang, target_lang)', 'n_train': 374, 'n_test': 94, 'positive_rate_train': 0.6016042780748663, 'positive_rate_test': 0.6063829787234043, 'ROC_AUC': 0.6014698909435752, 'PR_AUC': 0.6743797504246098, 'Precision': 0.71875, 'Recall': 0.40350877192982454, 'F1': 0.5168539325842697, 'TN': 28, 'FP': 9, 'FN': 34, 'TP': 23}, {'model': 'Model 2: (rich feature set)', 'n_train': 374, 'n_test': 94, 'positive_rate_train': 0.6016042780748663, 'positive_rate_test': 0.6063829787234043, 'ROC_AUC': 0.7477477477477478, 'PR_AUC': 0.827377860557543, 'Precision': 0.7413793103448276, 'Recall': 0.7543859649122807, 'F1': 0.7478260869565218, 'TN': 22, 'FP': 15, 'FN': 14, 'TP': 43}]